## 1. Imports and paths

In [1]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd

from pyBKT.models import Model

BASE_DIR = Path("..").resolve()          # app/
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = DATA_DIR / "outputs"
MODELS_DIR = BASE_DIR / "models"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("OUTPUTS_DIR:", OUTPUTS_DIR)
print("MODELS_DIR:", MODELS_DIR)


BASE_DIR: /mnt/c/Users/Melany Nuria/Desktop/TFG/adaptive_bayesian_its/backend_fastAPI/app
PROCESSED_DIR: /mnt/c/Users/Melany Nuria/Desktop/TFG/adaptive_bayesian_its/backend_fastAPI/app/data/processed
OUTPUTS_DIR: /mnt/c/Users/Melany Nuria/Desktop/TFG/adaptive_bayesian_its/backend_fastAPI/app/data/outputs
MODELS_DIR: /mnt/c/Users/Melany Nuria/Desktop/TFG/adaptive_bayesian_its/backend_fastAPI/app/models


## 2. Load cleaned dataset

In [2]:
train_path = PROCESSED_DIR / "df_train_kc_cleaned.csv"
test_path = PROCESSED_DIR / "df_test_kc_cleaned.csv"

df_train = pd.read_csv(train_path, encoding="latin")
df_test = pd.read_csv(test_path)

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)
print(df_train.columns.tolist())

Train shape: (1183869, 10)
Test shape: (2497, 13)
['Anon Student Id', 'Problem Name', 'Step Name', 'Step Start Time', 'Step End Time', 'Correct First Attempt', 'Incorrects', 'Hints', 'Corrects', 'skill_name']


## 3. Helpers and mapping

In [3]:
KC_NAME_MAP = {
    "Expand / Eliminate Parentheses": "expand_eliminate_parentheses",
    "Combine Like Terms": "combine_like_terms",
    "Move Constants Across the Equation": "move_constants",
    "Move Variable Terms to One Side": "move_variables_one_side",
    "Remove Coefficient": "remove_coefficient",
    "Normalize Negative Variable / Sign": "normalize_negative_sign",
}

In [4]:
def prepare_columns(df):
    df = df.copy()
    time_cols = [
        "Step Start Time",
        "First Transaction Time",
        "Correct Transaction Time",
        "Step End Time"
    ]
    for col in time_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    
    numeric_cols = [
        "Correct First Attempt",
        "Incorrects",
        "Hints",
        "Corrects"
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df
def build_pybkt_df(df):
    df = df.copy()

    # Keep only rows with the core fields
    df = df.dropna(subset=[
        "Anon Student Id",
        "skill_name",
        "Correct First Attempt",
        "Step Start Time"
    ]).copy()

    # Keep only binary observations
    df["Correct First Attempt"] = pd.to_numeric(df["Correct First Attempt"], errors="coerce")
    df = df[df["Correct First Attempt"].isin([0, 1])].copy()

    # Keep only final KCs
    df = df[df["skill_name"].isin(KC_NAME_MAP.keys())].copy()

    # Build standard BKT columns
    df["user_id"] = df["Anon Student Id"].astype(str)
    df["skill_name"] = df["skill_name"].map(KC_NAME_MAP)
    df["correct"] = df["Correct First Attempt"].astype(int)

    # Sort chronologically
    df["Step Start Time"] = pd.to_datetime(df["Step Start Time"], errors="coerce")
    df = df.sort_values(
        by=["user_id", "Step Start Time", "Problem Name", "Step Name"]
    ).reset_index(drop=True)

    # Unique order_id
    df["order_id"] = np.arange(1, len(df) + 1)

    # Minimal pyBKT dataframe
    out = df[["order_id", "user_id", "skill_name", "correct"]].copy()

    return df

In [5]:
df_train = prepare_columns(df_train)
train_bkt = build_pybkt_df(df_train)
train_bkt = train_bkt.sort_values(["user_id", "order_id"]).reset_index(drop=True)
print("train_bkt shape:", train_bkt.shape)

train_bkt shape: (1183869, 13)


In [6]:
model = Model(seed = 42, num_fits = 1)
model.fit(data = train_bkt)
print(model.params())

                                               value
skill                        param   class          
expand_eliminate_parentheses prior   default 0.75917
                             learns  default 0.00645
                             guesses default 0.74314
                             slips   default 0.07993
                             forgets default 0.00000
combine_like_terms           prior   default 0.77833
                             learns  default 0.00128
                             guesses default 0.80593
                             slips   default 0.06711
                             forgets default 0.00000
move_constants               prior   default 0.39090
                             learns  default 0.02206
                             guesses default 0.58181
                             slips   default 0.08468
                             forgets default 0.00000
remove_coefficient           prior   default 0.41019
                             learns  default 0

The estimated BKT parameters revealed unusually high guess probabilities for several skills, particularly Combine Like Terms, Expand / Eliminate Parentheses, and Remove Coefficient. This suggests that the current KC mapping and preprocessing may still group together heterogeneous step types or include opportunities that do not correspond to clean, independent demonstrations of the target skill. In addition, the low estimated learning probabilities for some KCs indicate that the model explains performance more through high initial knowledge or guessing than through observable learning over time. Therefore, although the fitted model provides a first approximation of student knowledge, the parameter values indicate that further refinement of the KC taxonomy and step filtering is necessary before drawing strong pedagogical conclusions.